In [1]:
!lscpu

import psutil

print("CPU cores:", psutil.cpu_count(logical=False))
print("Logical CPUs:", psutil.cpu_count(logical=True))
print("Memory (GB):", round(psutil.virtual_memory().total / 1e9, 2))
print("Disk space (GB):", round(psutil.disk_usage('/').total / 1e9, 2))

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      12
  On-line CPU(s) list:       0-11
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      6
    Socket(s):               1
    Stepping:                7
    BogoMIPS:                4400.37
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                 

In [4]:
import os, sys, time, argparse, json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ── Colab / A100 setup ────────────────────────────────────────────────────────
# Run in Colab with:
#   !pip install torchvision scikit-learn matplotlib
#   !python exp9_vit_stl10.py
#
# Optional: mount Drive first and set GDRIVE_DIR below for persistent storage.
# ─────────────────────────────────────────────────────────────────────────────
GDRIVE_DIR  = None    # override: e.g. '/content/drive/MyDrive/VJEPA/exp9'
_IN_COLAB   = os.path.isdir('/content')
_DRIVE_ROOT = '/content/drive/MyDrive'
_DRIVE_AVAIL = os.path.isdir(_DRIVE_ROOT)   # True if Drive is already mounted
STL10_DIR   = '/content/stl10' if _IN_COLAB else '/tmp/stl10'
USE_AMP     = True    # bfloat16 autocast — ~2–3× faster on A100, no GradScaler needed
# Auto-disable AMP if the GPU doesn't support bfloat16 (requires A100/H100)
_BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_AMP   = USE_AMP and _BF16_OK

try:
    import torchvision
    import torchvision.transforms as T
    import torchvision.transforms.functional as TF
    HAS_TV = True
except ImportError:
    HAS_TV = False
    sys.exit("torchvision not found. Run: pip install torchvision")

# ── Output directory: Drive (auto) > /content > local ─────────────────────────
def _resolve_out():
    if GDRIVE_DIR:
        return GDRIVE_DIR
    if _DRIVE_AVAIL:                                    # Drive already mounted
        return os.path.join(_DRIVE_ROOT, 'VJEPA_experiments', 'exp9_vit_stl10')
    if _IN_COLAB:
        return '/content/exp9_vit_stl10'
    else:
        # Fallback for non-Colab environment
        return os.path.join(os.getcwd(), 'new experiment results', 'exp9_vit_stl10')

OUT_DIR = _resolve_out()
os.makedirs(OUT_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────────
SEED         = 111
IMG_SIZE     = 64          # resize STL-10 96×96 → 64×64 for speed
PATCH_SIZE   = 8           # 64/8 = 8×8 = 64 patches
VIT_DIM      = 192         # token embedding dimension
VIT_DEPTH    = 4           # transformer blocks
VIT_HEADS    = 3           # attention heads (192/3=64 per head)
VIT_MLP_RATIO= 2           # MLP hidden = 2 × VIT_DIM
DIM_Z        = 64          # latent dimension (CLS → z)
PRED_HIDDEN  = 256

SCALES       = [0.0, 0.5, 1.0, 2.0]   # distractor multiplier
CLASS_REPEAT = 8                        # steps per class
N_SEQ        = 4000                     # training timesteps
N_SEQ_TEST   = 1000                     # test timesteps
N_EPOCHS     = 3000
LR           = 2e-4                     # smaller for ViT
BATCH        = 512                      # doubled for A100 (40 GB VRAM)
EMA_TAU      = 0.99
BETA         = 0.01                     # KL weight (VJEPA/BJEPA)
GAMMA        = 0.10                     # structural prior weight (BJEPA)
ES_PATIENCE  = 100                      # early-stop patience (epochs); ViT loss oscillates early

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True   # A100: auto-tune convolution kernels
print(f"Device: {device}  |  img={IMG_SIZE}×{IMG_SIZE}  |  DIM_Z={DIM_Z}  "
      f"|  ViT depth={VIT_DEPTH} dim={VIT_DIM} heads={VIT_HEADS}  AMP={USE_AMP}")

# ── STL-10 loader ──────────────────────────────────────────────────────────────
def load_stl10():
    """
    Load STL-10 labeled train and test splits.
    Returns numpy arrays: (imgs_tr, lbs_tr, imgs_te, lbs_te)
      imgs: float32 [N, 3, 64, 64], values in [0,1]
    """
    print("Loading STL-10...")
    tfm = T.Compose([T.Resize(IMG_SIZE), T.ToTensor()])
    os.makedirs(STL10_DIR, exist_ok=True)
    ds_tr = torchvision.datasets.STL10(STL10_DIR, split='train',
                                        download=True, transform=tfm)
    ds_te = torchvision.datasets.STL10(STL10_DIR, split='test',
                                        download=True, transform=tfm)
    # Stack into arrays
    imgs_tr = torch.stack([ds_tr[i][0] for i in range(len(ds_tr))]).numpy()  # [5000,3,64,64]
    lbs_tr  = np.array([ds_tr[i][1] for i in range(len(ds_tr))])
    imgs_te = torch.stack([ds_te[i][0] for i in range(len(ds_te))]).numpy()  # [8000,3,64,64]
    lbs_te  = np.array([ds_te[i][1] for i in range(len(ds_te))])
    print(f"STL-10 loaded: train={len(imgs_tr)}, test={len(imgs_te)}, "
          f"shape={imgs_tr.shape[1:]}")
    return imgs_tr, lbs_tr, imgs_te, lbs_te


def build_sequence(imgs, labels, n_steps, repeat=CLASS_REPEAT, rng=None):
    """
    Build a temporal STL-10 signal sequence (class cycling).
    Returns:
      x_sig [n_steps, 3, 64, 64],  y_sig [n_steps] (class 0–9)
    """
    if rng is None: rng = np.random.RandomState(SEED)
    by_class = {c: imgs[labels == c] for c in range(10)}
    x_sig, y_sig = [], []
    for step in range(n_steps):
        cls = (step // repeat) % 10
        pool = by_class[cls]
        idx = rng.randint(0, len(pool))
        x_sig.append(pool[idx])
        y_sig.append(cls)
    return np.stack(x_sig), np.array(y_sig, dtype=np.int64)


def build_distractor_sequence(imgs, n_steps, rng=None):
    """Sample n_steps random STL-10 images as distractors."""
    if rng is None: rng = np.random.RandomState(SEED + 1)
    idxs = rng.randint(0, len(imgs), size=n_steps)
    return imgs[idxs]   # [n_steps, 3, 64, 64]


def make_pairs(x_sig, x_dist, sigma):
    """
    Build (x_t, x_{t+1}, y_{t+1}) pairs from the temporal sequence.
    Observation: x = signal + sigma * distractor, clamped to [0,1].
    Returns tensors: xt [N,3,H,W], xn [N,3,H,W], yn [N] (next-frame class).
    """
    obs = np.clip(x_sig + sigma * x_dist, 0.0, 1.0)  # [N_SEQ, 3, H, W]
    xt = torch.FloatTensor(obs[:-1])   # [N_SEQ-1, ...]
    xn = torch.FloatTensor(obs[1:])
    return xt, xn


# ── ViT architecture ───────────────────────────────────────────────────────────
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(dim, 3 * dim, bias=False)
        self.proj      = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = (self.qkv(x)
                   .reshape(B, N, 3, self.num_heads, self.head_dim)
                   .permute(2, 0, 3, 1, 4))
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class ViTBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadSelfAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTEncoder(nn.Module):
    """
    ViT-Tiny encoder (from scratch).
    img_size=64, patch_size=8 → 8×8=64 patches.
    dim=192, depth=4, heads=3, mlp_ratio=2.
    CLS token → Linear(192, out_dim) → z.
    """
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, in_ch=3,
                 dim=VIT_DIM, depth=VIT_DEPTH, heads=VIT_HEADS,
                 mlp_ratio=VIT_MLP_RATIO, out_dim=DIM_Z):
        super().__init__()
        n_patches         = (img_size // patch_size) ** 2
        self.patch_embed  = nn.Conv2d(in_ch, dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token    = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed    = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        self.blocks       = nn.Sequential(*[ViTBlock(dim, heads, mlp_ratio)
                                            for _ in range(depth)])
        self.norm         = nn.LayerNorm(dim)
        self.head         = nn.Linear(dim, out_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)   # [B, N, dim]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.blocks(x)
        x = self.norm(x)
        return self.head(x[:, 0])   # CLS token → [B, DIM_Z]


# ── Model classes ──────────────────────────────────────────────────────────────
class JEPAViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc  = ViTEncoder()
        self.pred = nn.Sequential(nn.Linear(DIM_Z, PRED_HIDDEN), nn.ReLU(),
                                  nn.Linear(PRED_HIDDEN, DIM_Z))
        self.tgt  = deepcopy(self.enc)
        for p in self.tgt.parameters(): p.requires_grad_(False)

    def forward(self, xt, xn):
        zp = self.pred(self.enc(xt))
        with torch.no_grad(): zt = self.tgt(xn)
        return zp, zt

    def ema(self):
        for p, tp in zip(self.enc.parameters(), self.tgt.parameters()):
            tp.data.mul_(EMA_TAU).add_(p.data, alpha=1 - EMA_TAU)

    @torch.no_grad()
    def latent(self, x): return self.enc(x)

    @torch.no_grad()
    def pred_latent(self, x):
        """Predictor output — used as probe input for next-frame prediction."""
        return self.pred(self.enc(x))


class VJEPAViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc     = ViTEncoder()
        self.pred_mu = nn.Sequential(nn.Linear(DIM_Z, PRED_HIDDEN), nn.ReLU(),
                                     nn.Linear(PRED_HIDDEN, DIM_Z))
        self.pred_lv = nn.Sequential(nn.Linear(DIM_Z, PRED_HIDDEN), nn.ReLU(),
                                     nn.Linear(PRED_HIDDEN, DIM_Z))
        self.tgt     = deepcopy(self.enc)
        for p in self.tgt.parameters(): p.requires_grad_(False)

    def forward(self, xt, xn):
        z    = self.enc(xt)
        p_mu = self.pred_mu(z)
        p_lv = self.pred_lv(z).clamp(-10, 10)
        with torch.no_grad(): zt = self.tgt(xn)
        return p_mu, p_lv, zt

    def ema(self):
        for p, tp in zip(self.enc.parameters(), self.tgt.parameters()):
            tp.data.mul_(EMA_TAU).add_(p.data, alpha=1 - EMA_TAU)

    @torch.no_grad()
    def latent(self, x): return self.enc(x)

    @torch.no_grad()
    def pred_latent(self, x): return self.pred_mu(self.enc(x))

    @torch.no_grad()
    def pred_variance(self, x):
        return self.pred_lv(self.enc(x)).clamp(-10, 10).exp()   # [B, DIM_Z]


class BJEPAViT(VJEPAViT):
    """VJEPA-ViT + stationarity prior N(0,I) fused via Product of Experts."""

    @torch.no_grad()
    def latent(self, x):
        z     = self.enc(x)
        p_mu  = self.pred_mu(z)
        p_lv  = self.pred_lv(z).clamp(-10, 10)
        prec_d = (-p_lv).exp()
        prec   = prec_d + torch.ones_like(p_lv)
        return (p_mu * prec_d) / prec

    @torch.no_grad()
    def pred_latent(self, x):
        return self.latent(x)   # PoE-fused prediction


# ── Loss functions ─────────────────────────────────────────────────────────────
def vicreg(x, y, lam_sim=25., lam_std=25., lam_cov=1.):
    """VICReg loss (variance-invariance-covariance regularisation)."""
    loss = lam_sim * (x - y).pow(2).mean()
    for z in (x, y):
        std  = z.std(dim=0)
        loss += lam_std / 2 * torch.relu(1.0 - std).mean()
        z_c  = z - z.mean(dim=0)
        cov  = (z_c.T @ z_c) / (z.shape[0] - 1)
        off  = cov.pow(2)
        off.fill_diagonal_(0.0)
        loss += lam_cov * off.sum() / z.shape[1]
    return loss


def bjepa_kl(d_mu, d_lv, pr_mu=None, pr_lv=None):
    """KL( dynamics N(d_mu,exp(d_lv)) || stationarity prior N(0,I))."""
    # If no explicit prior params, use N(0,I)
    if pr_mu is None:
        return -0.5 * (1 + d_lv - d_mu.pow(2) - d_lv.exp()).sum(-1).mean()
    vr = (d_lv - pr_lv).exp()
    return 0.5 * (vr + (pr_mu - d_mu).pow(2) / pr_lv.exp() - 1 - (d_lv - pr_lv)).sum(-1).mean()


# ── Training ───────────────────────────────────────────────────────────────────
def train_model(model, name, xt, xn, n_epochs=None):
    """
    Train JEPA/VJEPA/BJEPA-ViT on paired observations (xt, xn).
    Uses VICReg for representation + stop-gradient NLL for uncertainty head.
    """
    if n_epochs is None:
        n_epochs = N_EPOCHS
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs,
                                                      eta_min=LR * 0.1)
    N = len(xt)
    best_loss = float('inf'); no_improve = 0

    for ep in range(n_epochs):
        model.train()
        epoch_loss = 0.0; n_batches = 0
        perm = torch.randperm(N)

        amp_ctx = torch.autocast('cuda', dtype=torch.bfloat16,
                                  enabled=USE_AMP and device.type == 'cuda')
        for i in range(0, N, BATCH):
            idx = perm[i:i + BATCH]
            x_t = xt[idx].to(device)
            x_n = xn[idx].to(device)

            with amp_ctx:
                if name == 'JEPA':
                    zp, zt = model(x_t, x_n)
                    loss   = vicreg(zp, zt)

                else:   # VJEPA or BJEPA (same training objective)
                    p_mu, p_lv, zt = model(x_t, x_n)
                    # Representation: VICReg on predictor mean vs EMA target
                    loss = vicreg(p_mu, zt)
                    # Uncertainty calibration: stop-gradient NLL on pred_lv
                    p_lv_sg = model.pred_lv(model.enc(x_t).detach()).clamp(-10, 10)
                    pv  = p_lv_sg.exp()
                    res = (zt.detach() - p_mu.detach()).pow(2)
                    loss = loss + 0.1 * 0.5 * (pv.log() + res / pv).sum(-1).mean()
                    # Structural KL for BJEPA
                    if name == 'BJEPA':
                        loss = loss + GAMMA * bjepa_kl(p_mu.detach(), p_lv.detach())

            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item(); n_batches += 1

        model.ema()   # EMA update once per epoch
        scheduler.step()

        avg = epoch_loss / n_batches
        if (ep + 1) % 100 == 0:
            print(f"      epoch {ep+1:4d}/{n_epochs}  loss={avg:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")
        if avg < best_loss - 1e-4:
            best_loss = avg; no_improve = 0
        else:
            no_improve += 1
            if no_improve >= ES_PATIENCE:
                print(f"      [early stop] epoch {ep+1}  best={best_loss:.4f}")
                break


# ── Evaluation ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def extract_pred_latents(model, x_seq):
    """
    Extract predicted next-frame latents z_pred for all steps.
    Returns [N-1, DIM_Z] (predict x_{t+1} from x_t).
    """
    model.eval()
    outs = []
    for i in range(0, len(x_seq) - 1, 256):
        batch = x_seq[i:i+256].to(device)
        outs.append(model.pred_latent(batch).cpu())
    model.train()
    # Trim to exactly N-1: the last batch may extend one frame past N-1
    # because the loop uses batches of 256 starting at i < N-1.
    N = len(x_seq)
    return torch.cat(outs, dim=0)[:N - 1]   # [N-1, DIM_Z]


def evaluate_accuracy(model, name,
                       x_sig_tr, y_sig_tr,
                       x_sig_te, y_sig_te,
                       x_dist_tr, x_dist_te, sigma):
    """
    Fit logistic probe on training predicted latents → next class label.
    Evaluate on test predicted latents.
    """
    # Build noisy observations
    obs_tr = np.clip(x_sig_tr + sigma * x_dist_tr, 0.0, 1.0)
    obs_te = np.clip(x_sig_te + sigma * x_dist_te, 0.0, 1.0)

    x_tr = torch.FloatTensor(obs_tr)
    x_te = torch.FloatTensor(obs_te)

    # Extract predicted latents (z_t → predict z_{t+1})
    z_pred_tr = extract_pred_latents(model, x_tr).numpy()   # [N-1, DIM_Z]
    z_pred_te = extract_pred_latents(model, x_te).numpy()

    # Labels: next-frame class
    y_tr = y_sig_tr[1:]   # [N-1]
    y_te = y_sig_te[1:]

    # Fit logistic probe
    clf = LogisticRegression(max_iter=2000, random_state=0, C=1.0,
                              solver='lbfgs', multi_class='auto')
    clf.fit(z_pred_tr, y_tr)
    acc_tr = clf.score(z_pred_tr, y_tr) * 100.0
    acc_te = clf.score(z_pred_te, y_te) * 100.0
    return acc_tr, acc_te


# ── Per-dim variance plot ──────────────────────────────────────────────────────
def plot_perdim_variance(vjepa, x_seq_tr, sigma, save_path):
    """Bar chart of per-dimension mean predictive variance for VJEPA-ViT."""
    obs = torch.FloatTensor(x_seq_tr)
    all_var = []
    vjepa.eval()
    with torch.no_grad():
        for i in range(0, len(obs), 256):
            b = obs[i:i+256].to(device)
            all_var.append(vjepa.pred_variance(b).cpu())
    vjepa.train()
    all_var  = torch.cat(all_var).numpy()   # [N, DIM_Z]
    dim_mean = all_var.mean(0)
    dim_std  = all_var.std(0)
    order    = np.argsort(dim_mean)

    fig, ax = plt.subplots(figsize=(10, 4))
    xs = np.arange(DIM_Z)
    colours = ['#2196F3' if v < np.median(dim_mean) else '#FF5722'
               for v in dim_mean[order]]
    ax.bar(xs, dim_mean[order], yerr=dim_std[order], color=colours,
           edgecolor='white', linewidth=0.4, capsize=2,
           error_kw={'elinewidth': 0.8})
    ax.axhline(np.median(dim_mean), color='k', linestyle='--', linewidth=1.2,
               label='Median variance')
    ax.set_xlabel('Latent dimension (sorted by mean variance)', fontsize=11)
    ax.set_ylabel('Mean predictive variance $\\mathbb{E}[\\exp(\\ell_\\phi)]$', fontsize=11)
    ax.set_title(f'Exp. 9: VJEPA-ViT Per-Dimension Predictive Variance  (σ={sigma})\n'
                 'Blue = signal dims (low, predictable);  Red = noise dims (high, uncertain)',
                 fontsize=10)
    ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")


# ── Incremental Drive save ────────────────────────────────────────────────────
def save_incremental(results, scales_done):
    """
    Write results_table.json + results_table.csv after every (σ, model) combo.
    Survives Colab disconnection — partial results are always on Drive.

    results_table.csv layout (copy directly into paper table):
      sigma | JEPA_test | VJEPA_test | BJEPA_test | JEPA_train | VJEPA_train | BJEPA_train
    """
    names = ['JEPA', 'VJEPA', 'BJEPA']

    # JSON — full (train, test) pairs
    blob = {}
    for s in scales_done:
        blob[str(s)] = {n: list(results[s][n]) for n in names if n in results[s]}
    json_path = os.path.join(OUT_DIR, 'results_table.json')
    with open(json_path, 'w') as f:
        json.dump(blob, f, indent=2)

    # CSV — test accuracy first (what goes in the paper table), then train
    header = 'sigma,' + ','.join(f'{n}_test' for n in names) + ',' \
                       + ','.join(f'{n}_train' for n in names)
    rows = [header]
    for s in scales_done:
        te_vals = [f"{results[s][n][1]:.2f}" if n in results[s] else '' for n in names]
        tr_vals = [f"{results[s][n][0]:.2f}" if n in results[s] else '' for n in names]
        rows.append(f"{s}," + ','.join(te_vals) + ',' + ','.join(tr_vals))
    csv_path = os.path.join(OUT_DIR, 'results_table.csv')
    with open(csv_path, 'w') as f:
        f.write('\n'.join(rows) + '\n')

    # Human-readable summary printed to stdout (visible in Colab cell output)
    print(f"\n  ┌─ Accuracy table so far (test %) — saved to {OUT_DIR} ─────────────")
    print(f"  │  {'σ':>5}  " + "  ".join(f"{n:>10}" for n in names))
    for s in scales_done:
        vals = "  ".join(
            f"{results[s][n][1]:>9.1f}%" if n in results[s] else f"{'—':>10}"
            for n in names)
        print(f"  │  {s:>5.1f}  {vals}")
    print(f"  └─ Chance = 10.0%  (10 classes)")
    print(f"  [Saved] {json_path}")
    print(f"  [Saved] {csv_path}\n")


# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--scales', nargs='+', type=float, default=SCALES,
                        help='Distractor noise scales to run')
    parser.add_argument('--dry-run', action='store_true',
                        help='Run 5 epochs only to verify the loop')
    # Fix: tell argparse to ignore Colab kernel arguments
    args = parser.parse_args(args=[])

    n_epochs = 5 if args.dry_run else N_EPOCHS

    # Load STL-10
    imgs_tr, lbs_tr, imgs_te, lbs_te = load_stl10()

    # Build sequences (reuse across scales — only observations change)
    rng_sig_tr  = np.random.RandomState(SEED)
    rng_dist_tr = np.random.RandomState(SEED + 1)
    rng_sig_te  = np.random.RandomState(SEED + 10)
    rng_dist_te = np.random.RandomState(SEED + 11)

    x_sig_tr,  y_sig_tr  = build_sequence(imgs_tr, lbs_tr, N_SEQ,      rng=rng_sig_tr)
    x_dist_tr             = build_distractor_sequence(imgs_tr, N_SEQ,   rng=rng_dist_tr)
    x_sig_te,  y_sig_te  = build_sequence(imgs_te, lbs_te, N_SEQ_TEST,  rng=rng_sig_te)
    x_dist_te             = build_distractor_sequence(imgs_te, N_SEQ_TEST, rng=rng_dist_te)

    print(f"\nSequences built: train={N_SEQ}, test={N_SEQ_TEST}, classes=10, "
          f"repeat_K={CLASS_REPEAT}")

    # ── Results table ──────────────────────────────────────────────────────────
    # results[sigma][model] = (train_acc, test_acc)
    results = {s: {} for s in args.scales}
    vjepa_model_sigma1 = None   # save for per-dim plot

    for sigma in args.scales:
        print(f"\n{'='*60}")
        print(f"  σ = {sigma}  (distractor scale)")
        print(f"{'='*60}")

        xt, xn = make_pairs(x_sig_tr, x_dist_tr, sigma)
        print(f"  Training pairs: {len(xt)}")

        for name, ModelClass in [('JEPA', JEPAViT),
                                  ('VJEPA', VJEPAViT),
                                  ('BJEPA', BJEPAViT)]:
            print(f"\n  Training {name}-ViT ...")
            t0 = time.time()
            model = ModelClass().to(device)
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"    Parameters: {n_params:,}")

            train_model(model, name, xt, xn, n_epochs=n_epochs)

            print(f"    Training time: {time.time()-t0:.1f}s")

            acc_tr, acc_te = evaluate_accuracy(
                model, name,
                x_sig_tr, y_sig_tr, x_sig_te, y_sig_te,
                x_dist_tr, x_dist_te, sigma)
            results[sigma][name] = (acc_tr, acc_te)
            print(f"    {name}-ViT  train={acc_tr:.1f}%  test={acc_te:.1f}%")

            # ── Flush results to Drive after every (σ, model) ─────────────────
            # Gives a complete partial table even if Colab disconnects mid-run.
            completed_scales = [s for s in args.scales if results[s]]
            save_incremental(results, completed_scales)

            if name == 'VJEPA' and sigma == 1.0:
                vjepa_model_sigma1 = model

    # ── Print results table ────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  Exp. 9: VJEPA-ViT on STL-10 — Next-Class Accuracy (test %)")
    print(f"{'='*65}")
    header = f"  {'σ':>5}  " + "  ".join(f"{n:>12}" for n in ['JEPA-ViT', 'VJEPA-ViT', 'BJEPA-ViT'])
    print(header)
    print("  " + "-" * 55)
    for sigma in args.scales:
        row = f"  {sigma:>5.1f}  "
        for name in ['JEPA', 'VJEPA', 'BJEPA']:
            if name in results[sigma]:
                acc = results[sigma][name][1]
                row += f"  {acc:>12.1f}%"
            else:
                row += f"  {'—':>12}"
        print(row)
    print(f"\n  Chance = 10.0%  (10 classes)")
    print(f"{'='*65}")

    # ── Accuracy plot ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))
    colours = {'JEPA': '#1f77b4', 'VJEPA': '#ff7f0e', 'BJEPA': '#9467bd'}
    markers = {'JEPA': 'o',       'VJEPA': 's',        'BJEPA': '^'}
    for name in ['JEPA', 'VJEPA', 'BJEPA']:
        ys = [results[s][name][1] if name in results[s] else np.nan
              for s in args.scales]
        ax.plot(args.scales, ys, marker=markers[name], linewidth=2.5,
                markersize=9, color=colours[name], label=f'{name}-ViT')
    ax.axhline(10.0, color='gray', linestyle='--', linewidth=1.2, label='Chance (10%)')
    ax.set_xlabel('Distractor scale σ', fontsize=12)
    ax.set_ylabel('Next-class test accuracy (%)', fontsize=12)
    ax.set_title('Exp. 9: VJEPA-ViT Scalability on STL-10\n'
                 'ViT-Tiny encoder (from scratch, patch=8, depth=4, dim=192)',
                 fontsize=11)
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    ax.set_xticks(args.scales)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    fig_path = os.path.join(OUT_DIR, 'fig_vit_accuracy.pdf')
    plt.savefig(fig_path, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {fig_path}")

    # ── Per-dim variance plot ─────────────────────────────────────────────────
    if vjepa_model_sigma1 is not None:
        obs_tr = np.clip(x_sig_tr + 1.0 * x_dist_tr, 0.0, 1.0)
        perdim_path = os.path.join(OUT_DIR, 'fig_vit_perdim.pdf')
        plot_perdim_variance(vjepa_model_sigma1,
                             obs_tr, sigma=1.0,
                             save_path=perdim_path)

    # ── Save numerical results ────────────────────────────────────────────────
    np.save(os.path.join(OUT_DIR, 'results_vit_stl10.npy'),
            {str(s): {n: results[s][n] for n in results[s]}
             for s in results})
    print(f"\nResults saved to: {OUT_DIR}")


if __name__ == '__main__':
    main()

Device: cuda  |  img=64×64  |  DIM_Z=64  |  ViT depth=4 dim=192 heads=3  AMP=True
Loading STL-10...


100%|██████████| 2.64G/2.64G [04:57<00:00, 8.87MB/s]


STL-10 loaded: train=5000, test=8000, shape=(3, 64, 64)

Sequences built: train=4000, test=1000, classes=10, repeat_K=8

  σ = 0.0  (distractor scale)
  Training pairs: 3999

  Training JEPA-ViT ...
    Parameters: 1,281,344
      epoch  100/3000  loss=19.8504  lr=2.00e-04
      epoch  200/3000  loss=24.3670  lr=1.98e-04
      [early stop] epoch 206  best=19.7250
    Training time: 60.8s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


    JEPA-ViT  train=42.9%  test=25.0%

  ┌─ Accuracy table so far (test %) — saved to /content/exp9_vit_stl10 ─────────────
  │      σ        JEPA       VJEPA       BJEPA
  │    0.0       25.0%           —           —
  └─ Chance = 10.0%  (10 classes)
  [Saved] /content/exp9_vit_stl10/results_table.json
  [Saved] /content/exp9_vit_stl10/results_table.csv


  Training VJEPA-ViT ...
    Parameters: 1,314,432
      epoch  100/3000  loss=14.6629  lr=2.00e-04
      epoch  200/3000  loss=14.8805  lr=1.98e-04
      [early stop] epoch 245  best=13.5709
    Training time: 82.1s


...

    JEPA-ViT  train=20.8%  test=11.5%

  ┌─ Accuracy table so far (test %) — saved to /content/exp9_vit_stl10 ─────────────
  │      σ        JEPA       VJEPA       BJEPA
  │    0.0       25.0%       23.3%       23.0%
  │    0.5       16.6%       17.8%       19.7%
  │    1.0       11.3%       12.2%       17.2%
  │    2.0       11.5%           —           —
  └─ Chance = 10.0%  (10 classes)
  [Saved] /content/exp9_vit_stl10/results_table.json
  [Saved] /content/exp9_vit_stl10/results_table.csv


  Training VJEPA-ViT ...
    Parameters: 1,314,432
      epoch  100/3000  loss=18.4867  lr=2.00e-04
      [early stop] epoch 124  best=14.6771
    Training time: 41.3s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


    VJEPA-ViT  train=18.1%  test=14.5%

  ┌─ Accuracy table so far (test %) — saved to /content/exp9_vit_stl10 ─────────────
  │      σ        JEPA       VJEPA       BJEPA
  │    0.0       25.0%       23.3%       23.0%
  │    0.5       16.6%       17.8%       19.7%
  │    1.0       11.3%       12.2%       17.2%
  │    2.0       11.5%       14.5%           —
  └─ Chance = 10.0%  (10 classes)
  [Saved] /content/exp9_vit_stl10/results_table.json
  [Saved] /content/exp9_vit_stl10/results_table.csv


  Training BJEPA-ViT ...
    Parameters: 1,314,432
      epoch  100/3000  loss=23.9112  lr=2.00e-04
      [early stop] epoch 109  best=22.0975
    Training time: 36.4s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


    BJEPA-ViT  train=17.5%  test=13.4%

  ┌─ Accuracy table so far (test %) — saved to /content/exp9_vit_stl10 ─────────────
  │      σ        JEPA       VJEPA       BJEPA
  │    0.0       25.0%       23.3%       23.0%
  │    0.5       16.6%       17.8%       19.7%
  │    1.0       11.3%       12.2%       17.2%
  │    2.0       11.5%       14.5%       13.4%
  └─ Chance = 10.0%  (10 classes)
  [Saved] /content/exp9_vit_stl10/results_table.json
  [Saved] /content/exp9_vit_stl10/results_table.csv


  Exp. 9: VJEPA-ViT on STL-10 — Next-Class Accuracy (test %)
      σ      JEPA-ViT     VJEPA-ViT     BJEPA-ViT
  -------------------------------------------------------
    0.0            25.0%          23.3%          23.0%
    0.5            16.6%          17.8%          19.7%
    1.0            11.3%          12.2%          17.2%
    2.0            11.5%          14.5%          13.4%

  Chance = 10.0%  (10 classes)

Saved: /content/exp9_vit_stl10/fig_vit_accuracy.pdf
Saved: /content/exp9_vit_